# PCA on the Wine Dataset — Machine Learning Analysis

## Introduction

Principal Component Analysis (PCA) is a dimensionality-reduction technique that converts a set of related variables into a smaller number of principal components.

In this activity, PCA is applied to the Wine dataset after standardizing the features. The reduced representation is then compared with the original feature space to understand how dimensionality reduction affects the classification workflow.


## Objectives

The main objectives of this activity are:

- To inspect the Wine dataset before applying PCA.
- To understand the need for feature scaling before dimensionality reduction.
- To apply PCA and examine the variance retained by its components.
- To visualize the observations using principal components.
- To compare Logistic Regression with and without PCA.
- To examine the contribution of the original features to the principal components.


## 1. Importing the Required Libraries

The libraries below are used for data handling, visualization, preprocessing, PCA and classification evaluation.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score


## 2. Loading and Inspecting the Dataset

The Wine dataset is loaded from Scikit-learn. It contains numerical chemical measurements and a categorical target representing the wine classes.

A quick inspection is performed before preprocessing so that the dimensions and available classes are clear.


In [ ]:
wine = load_wine(as_frame=True)

X = wine.data.copy()
y = wine.target.copy()

print("Dataset shape:", X.shape)
print("Number of classes:", len(wine.target_names))
print("Class names:", list(wine.target_names))

display(X.head())


### Observation

The dataset contains 178 observations and 13 numerical features. Since the target represents multiple wine classes, this is a multi-class classification problem.


## 3. Preparing the Data

The feature matrix and target are divided into training and testing sets before scaling and PCA are fitted.

Keeping the test set separate is important because preprocessing parameters should be learned from the training data rather than from observations that will later be used for evaluation.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)


### Observation

The data was divided into 142 training observations and 36 testing observations. The test set remains separate so that the final model evaluation is based on data not used during training.


## 4. Feature Scaling

PCA is affected by the scale of the input variables. StandardScaler is therefore used so that each feature is centered and scaled before the principal components are calculated.


In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


### Observation

The scaler is fitted only on the training set and then applied to the test set. This avoids allowing information from the test data to influence the preprocessing stage.


## 5. Principal Component Analysis

PCA is first fitted without restricting the number of components. This allows the cumulative explained variance to be examined and helps determine how many components are needed to retain most of the information in the standardized features.


In [ ]:
pca_full = PCA()

X_train_pca_full = pca_full.fit_transform(X_train_scaled)
X_test_pca_full = pca_full.transform(X_test_scaled)

explained_variance = pca_full.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance)

variance_table = pd.DataFrame({
    "Component": np.arange(1, len(explained_variance) + 1),
    "Explained Variance": explained_variance,
    "Cumulative Variance": cumulative_variance
})

display(variance_table)


### 5.1 Explained Variance

The explained-variance ratios indicate how much of the variation in the standardized data is captured by each component. The cumulative values are used to identify the number of components required to retain at least 95% of the variance.


In [ ]:
variance_target = 0.95

n_components_95 = np.searchsorted(
    cumulative_variance,
    variance_target
) + 1

print(
    f"Components required to retain at least "
    f"{variance_target:.0%} variance: {n_components_95}"
)


### Observation

10 principal components are required to retain at least 95% of the variance. Together, these components retain approximately 96.2% of the variation in the standardized training data.


## 6. PCA Representation

A PCA transformation using the selected number of components is created for both the training and test sets. The transformation is fitted on the training data and then applied to the test data.


In [ ]:
pca = PCA(n_components=n_components_95)

X_train_reduced = pca.fit_transform(X_train_scaled)
X_test_reduced = pca.transform(X_test_scaled)

print("Original number of features:", X_train.shape[1])
print("PCA components retained:", X_train_reduced.shape[1])


### Observation

The original feature space contains 13 features, while the PCA representation retains 10 components. This reduces the dimensionality while meeting the selected 95% variance criterion.


## 7. Visualizing the First Two Principal Components

The first two principal components provide a two-dimensional view of the standardized data. This visualization is useful for seeing whether observations from the different wine classes form distinguishable groups in the reduced space.


In [ ]:
pca_2d = PCA(n_components=2)

X_train_2d = pca_2d.fit_transform(X_train_scaled)

plt.figure(figsize=(8, 6))

scatter = plt.scatter(
    X_train_2d[:, 0],
    X_train_2d[:, 1],
    c=y_train,
    alpha=0.8
)

plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title("Wine Dataset in the First Two Principal Components")
plt.colorbar(scatter, label="Wine Class")

plt.show()


### Observation

The first two principal components together explain approximately 55.1% of the variance in the standardized training data. The plot still shows a clear visual separation between the three wine classes, although the two-dimensional view does not represent all of the information retained by the full PCA transformation.


## 8. Classification Without PCA

Logistic Regression is first trained using the original standardized features. This provides a baseline against which the PCA-based model can be compared.


In [ ]:
model_without_pca = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=2000))
])

model_without_pca.fit(X_train, y_train)

pred_without_pca = model_without_pca.predict(X_test)

accuracy_without_pca = accuracy_score(
    y_test,
    pred_without_pca
)

print(
    "Accuracy without PCA:",
    round(accuracy_without_pca, 4)
)


## 9. Classification With PCA

The same Logistic Regression approach is evaluated using the PCA representation. The PCA and scaling steps are fitted only on the training data before predictions are generated for the test set.


In [ ]:
model_with_pca = LogisticRegression(max_iter=2000)

model_with_pca.fit(
    X_train_reduced,
    y_train
)

pred_with_pca = model_with_pca.predict(
    X_test_reduced
)

accuracy_with_pca = accuracy_score(
    y_test,
    pred_with_pca
)

print(
    "Accuracy with PCA:",
    round(accuracy_with_pca, 4)
)


### 9.1 Comparing the Two Approaches

The two accuracies are placed side by side to make the effect of dimensionality reduction easier to interpret.


In [ ]:
comparison = pd.DataFrame({
    "Approach": [
        "Original Features",
        "PCA Features"
    ],
    "Accuracy": [
        accuracy_without_pca,
        accuracy_with_pca
    ]
})

display(comparison)


### Observation

Logistic Regression achieves an accuracy of 0.972 using the original standardized features and 0.972 using the PCA representation. The two approaches have very similar test accuracy. This shows how much classification performance is retained after reducing the feature space.


## 10. PCA Loadings

PCA loadings indicate how strongly the original features contribute to each principal component. Examining the loadings can help interpret which original variables have greater influence on the reduced representation.


In [ ]:
loadings = pd.DataFrame(
    pca.components_.T,
    index=X.columns,
    columns=[
        f"PC{i + 1}"
        for i in range(n_components_95)
    ]
)

display(loadings)


### Observation

The loading values show how the original features contribute to the retained components. For example, `flavanoids` has the largest absolute loading on PC1, while `color_intensity` has the largest absolute loading on PC2. This shows that different original variables contribute differently to the directions captured by PCA.


## 11. Overall Interpretation

PCA reduced the feature space from 13 original variables to 10 components while retaining about 96.2% of the variance. The first two components alone explain about 55.1%, and the class clusters are visibly separated in the two-dimensional plot.

For the Logistic Regression comparison, both the original-feature and PCA approaches achieved the same test accuracy of 97.2%. In this experiment, PCA therefore reduced the dimensionality without reducing the observed classification accuracy.


## 12. Conclusion

This activity demonstrated a complete PCA workflow using the Wine dataset. The 13 original features were standardized using the training data, and PCA reduced them to 10 components while retaining approximately 96.2% of the variance.

The first two components explained about 55.1% of the variance and showed clear separation between the wine classes. Logistic Regression achieved 97.2% test accuracy both with the original features and with the PCA representation.

Therefore, in this experiment, PCA provided a lower-dimensional representation while maintaining the same observed classification accuracy as the original feature set.


## 13. Practice Questions

The required practice questions from the activity are included below.


## Practice questions

1. Why must features generally be standardized before PCA?
2. What does explained variance mean?
3. Why can PCA improve efficiency but sometimes reduce interpretability?
4. Compare accuracy with and without PCA.
5. How would you choose the number of components?
